In [3]:
import pandas as pd, re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords"); nltk.download("wordnet")

df = pd.read_parquet("../data/processed/reviews_clean.parquet")

stop = set(stopwords.words("english")) | {
    "book", "one", "would", "get", "also", "even", "much", "really",
    "like", "use", "time", "make", "good", "great", "well", "read",
    "product", "bought", "buy", "got", "could", "two", "first", "work",
}
lem = WordNetLemmatizer()

def tokens_for_lda(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [lem.lemmatize(w) for w in words if len(w) > 2 and w not in stop]

df["tokens"] = df["text"].map(tokens_for_lda)
print(df["tokens"].head(3).tolist())
print(df["tokens"].map(len).describe())
import os
tokens_path = "../data/processed/tokens.parquet"
if not os.path.exists(tokens_path):
    df[["review_id", "tokens"]].to_parquet(tokens_path, index=False)
    print(f"Saved tokens to {tokens_path}")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\neelg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\neelg\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


[['game', 'horrible', 'wait', 'long', 'anything', 'dog', 'game', 'boring', 'compared', 'nintendogs', 'save', 'money', 'game', 'birthday', 'wish', 'return', 'something', 'amusing'], ['lost', 'pennsylvania', 'without', 'doubt', 'worst', 'trail', 'guide', 'ever', 'used', 'done', 'hike', 'afraid', 'try', 'found', 'distance', 'wrong', 'landmark', 'wrong', 'trail', 'name', 'wrong', 'hike', 'lost', 'nearly', 'hour', 'able', 'locate', 'cut', 'trail', 'point', 'supposedly', 'marked', 'intersection', 'second', 'companion', 'spent', 'hour', 'looking', 'intersection', 'poorly', 'described', 'hike', 'included', 'rather', 'unattractive', 'segment', 'trough', 'parking', 'lot', 'along', 'logging', 'trail', 'last', 'summer', 'hiked', 'poland', 'found', 'utility', 'trail', 'guide', 'written', 'polish', 'speak', 'several', 'pennsylvania', 'trail', 'guide'], ['best', 'movie', 'ever', 'seen', 'watched', 'movie', 'least', 'time', 'nearly', 'know', 'word', 'word', 'looking', 'year', 'although', 'never', 'abl

In [1]:
import sys, gensim
print(sys.version)
print(gensim.__version__)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
4.4.0


In [5]:
import pandas as pd, re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords"); nltk.download("wordnet")

df = pd.read_parquet("../data/processed/reviews_clean.parquet")

stop = set(stopwords.words("english")) | {
    "book", "one", "would", "get", "also", "even", "much", "really",
    "like", "use", "time", "make", "good", "great", "well", "read",
    "product", "bought", "buy", "got", "could", "two", "first", "work",
}
lem = WordNetLemmatizer()

def tokens_for_lda(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [lem.lemmatize(w) for w in words if len(w) > 2 and w not in stop]

df["tokens"] = df["text"].map(tokens_for_lda)
print(df["tokens"].head(3).tolist())
print(df["tokens"].map(len).describe())
import os
tokens_path = "../data/processed/tokens.parquet"
if not os.path.exists(tokens_path):
    df[["review_id", "tokens"]].to_parquet(tokens_path, index=False)
    print(f"Saved tokens to {tokens_path}")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\neelg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\neelg\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


[['game', 'horrible', 'wait', 'long', 'anything', 'dog', 'game', 'boring', 'compared', 'nintendogs', 'save', 'money', 'game', 'birthday', 'wish', 'return', 'something', 'amusing'], ['lost', 'pennsylvania', 'without', 'doubt', 'worst', 'trail', 'guide', 'ever', 'used', 'done', 'hike', 'afraid', 'try', 'found', 'distance', 'wrong', 'landmark', 'wrong', 'trail', 'name', 'wrong', 'hike', 'lost', 'nearly', 'hour', 'able', 'locate', 'cut', 'trail', 'point', 'supposedly', 'marked', 'intersection', 'second', 'companion', 'spent', 'hour', 'looking', 'intersection', 'poorly', 'described', 'hike', 'included', 'rather', 'unattractive', 'segment', 'trough', 'parking', 'lot', 'along', 'logging', 'trail', 'last', 'summer', 'hiked', 'poland', 'found', 'utility', 'trail', 'guide', 'written', 'polish', 'speak', 'several', 'pennsylvania', 'trail', 'guide'], ['best', 'movie', 'ever', 'seen', 'watched', 'movie', 'least', 'time', 'nearly', 'know', 'word', 'word', 'looking', 'year', 'although', 'never', 'abl

In [6]:
from gensim.corpora import Dictionary

texts = df["tokens"].tolist()
dictionary = Dictionary(texts)
print("Before filtering:", len(dictionary))

# drop very rare words (in <10 reviews) and very common ones (in >40% of reviews)
dictionary.filter_extremes(no_below=10, no_above=0.4)
print("After filtering:", len(dictionary))

corpus = [dictionary.doc2bow(t) for t in texts]
print(corpus[0][:10])   # (word_id, count) pairs for review 0

Before filtering: 69627
After filtering: 11395
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 3), (7, 1), (8, 1), (9, 1)]


In [7]:
from gensim.models import LdaMulticore, CoherenceModel

results = {}
for k in [5, 10, 15, 20]:
    lda = LdaMulticore(
        corpus=corpus, id2word=dictionary, num_topics=k,
        passes=5, workers=3, random_state=42,
    )
    cm = CoherenceModel(model=lda, texts=texts, dictionary=dictionary,
                        coherence="c_npmi")
    score = cm.get_coherence()
    results[k] = (lda, score)
    print(f"k={k}: NPMI coherence = {score:.4f}")

k=5: NPMI coherence = 0.0272
k=10: NPMI coherence = 0.0420
k=15: NPMI coherence = 0.0305
k=20: NPMI coherence = 0.0278


In [8]:
best_k = max(results, key=lambda k: results[k][1])
print("Best k:", best_k)
for t in results[best_k][0].show_topics(num_topics=best_k, num_words=10, formatted=False):
    print(t[0], [w for w, _ in t[1]])

Best k: 10
0 ['story', 'character', 'book', 'reading', 'life', 'novel', 'love', 'think', 'page', 'author']
1 ['movie', 'dvd', 'watch', 'see', 'love', 'show', 'video', 'film', 'bad', 'seen']
2 ['best', 'series', 'movie', 'film', 'performance', 'story', 'funny', 'action', 'love', 'ever']
3 ['author', 'many', 'history', 'information', 'life', 'book', 'people', 'written', 'way', 'world']
4 ['game', 'old', 'fun', 'play', 'year', 'love', 'kid', 'toy', 'son', 'daughter']
5 ['album', 'song', 'music', 'sound', 'band', 'best', 'love', 'track', 'rock', 'listen']
6 ['work', 'sound', 'better', 'need', 'phone', 'problem', 'quality', 'little', 'bottle', 'using']
7 ['film', 'money', 'waste', 'bad', 'movie', 'star', 'worst', 'review', 'terrible', 'ever']
8 ['version', 'disc', 'copy', 'quality', 'set', 'dvd', 'edition', 'original', 'amazon', 'ray']
9 ['month', 'item', 'year', 'price', 'used', 'amazon', 'back', 'day', 'problem', 'purchased']


In [10]:
best_lda = results[best_k][0]
best_lda.save("../models/lda_k10.model")

In [11]:
for t in results[15][0].show_topics(num_topics=15, num_words=10, formatted=False):
    print(t[0], [w for w, _ in t[1]])

0 ['story', 'character', 'book', 'reading', 'novel', 'life', 'author', 'page', 'plot', 'think']
1 ['dvd', 'video', 'quality', 'player', 'version', 'sound', 'disc', 'movie', 'watch', 'ray']
2 ['series', 'best', 'fan', 'better', 'story', 'light', 'dark', 'made', 'end', 'vampire']
3 ['author', 'history', 'life', 'many', 'book', 'written', 'people', 'information', 'reading', 'way']
4 ['game', 'fun', 'play', 'old', 'year', 'toy', 'kid', 'son', 'playing', 'still']
5 ['love', 'child', 'year', 'old', 'friend', 'family', 'kid', 'little', 'life', 'boy']
6 ['bottle', 'water', 'filter', 'work', 'pan', 'better', 'used', 'need', 'iron', 'power']
7 ['money', 'waste', 'review', 'bad', 'star', 'worst', 'terrible', 'poor', 'better', 'save']
8 ['set', 'received', 'ordered', 'fit', 'box', 'amazon', 'size', 'price', 'item', 'cover']
9 ['month', 'year', 'used', 'day', 'back', 'coffee', 'unit', 'easy', 'made', 'problem']
10 ['love', 'food', 'dog', 'easy', 'recipe', 'best', 'day', 'cat', 'little', 'lot']
11 [

In [13]:
results[10][0].save("../models/lda_k10.model")
results[15][0].save("../models/lda_k15.model")